# Day 3: Estimation for Causal Effects

This notebook illustrates three estimators for the average treatment effect (ATE):

1. g-computation;
2. inverse probability weighting (IPW);
3. augmented inverse probability weighting (AIPW).


## Setup

We simulate an observational study with baseline covariates $W$, binary treatment $A$, and continuous outcome $Y$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold

rng = np.random.default_rng(123)

In [ ]:
def simulate_data(n=2000, seed=123):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(size=n)
    W2 = rng.binomial(1, 0.5, size=n)
    W3 = rng.normal(size=n)

    pA = expit(-0.2 + 0.8 * W1 - 0.6 * W2 + 0.4 * W3)
    A = rng.binomial(1, pA)

    tau = 1.0 + 0.5 * W2
    mu0 = 1.0 + W1 + 0.5 * W2 - 0.25 * W3
    Y = mu0 + tau * A + rng.normal(scale=1.0, size=n)

    return pd.DataFrame({
        'W1': W1,
        'W2': W2,
        'W3': W3,
        'A': A,
        'Y': Y,
        'pA_true': pA,
        'tau_true': tau,
        'mu0_true': mu0,
    })

df = simulate_data()
df.head()

In [ ]:
true_ate = df['tau_true'].mean()
print(f'True ATE in this simulated population: {true_ate:.3f}')

## Nuisance models

We estimate the outcome regression $Q(a,w)=E(Y\mid A=a,W=w)$ and propensity score $g(w)=P(A=1\mid W=w)$.

In [ ]:
W_cols = ['W1', 'W2', 'W3']
X_outcome = df[['A'] + W_cols]
X_propensity = df[W_cols]
Y = df['Y'].to_numpy()
A = df['A'].to_numpy()

q_model = LinearRegression().fit(X_outcome, Y)
g_model = LogisticRegression(max_iter=1000).fit(X_propensity, A)

X1 = df[W_cols].copy()
X1.insert(0, 'A', 1)
X0 = df[W_cols].copy()
X0.insert(0, 'A', 0)

q1_hat = q_model.predict(X1)
q0_hat = q_model.predict(X0)
g_hat = g_model.predict_proba(X_propensity)[:, 1]
g_hat = np.clip(g_hat, 0.01, 0.99)

## Estimators

In [ ]:
# 1. G-computation
ate_gcomp = np.mean(q1_hat - q0_hat)

# 2. Inverse probability weighting
ate_ipw = np.mean(A * Y / g_hat - (1 - A) * Y / (1 - g_hat))

# 3. Augmented inverse probability weighting
pseudo_outcome = (
    q1_hat - q0_hat
    + A * (Y - q1_hat) / g_hat
    - (1 - A) * (Y - q0_hat) / (1 - g_hat)
)
ate_aipw = np.mean(pseudo_outcome)
se_aipw = np.std(pseudo_outcome, ddof=1) / np.sqrt(len(df))

results = pd.DataFrame({
    'estimator': ['truth', 'g-computation', 'IPW', 'AIPW'],
    'estimate': [true_ate, ate_gcomp, ate_ipw, ate_aipw],
})
results

In [ ]:
print(f'AIPW 95% Wald CI: ({ate_aipw - 1.96 * se_aipw:.3f}, {ate_aipw + 1.96 * se_aipw:.3f})')

## Diagnostics: propensity scores and weights

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(g_hat[A == 1], bins=30, alpha=0.6, label='A=1')
plt.hist(g_hat[A == 0], bins=30, alpha=0.6, label='A=0')
plt.xlabel('Estimated propensity score')
plt.ylabel('Count')
plt.legend()
plt.title('Propensity-score overlap diagnostic')
plt.show()

In [ ]:
weights = A / g_hat + (1 - A) / (1 - g_hat)
pd.Series(weights).describe(percentiles=[0.5, 0.9, 0.95, 0.99])

## Cross-fitted AIPW estimator

The next function implements a simple two-fold or five-fold cross-fitted AIPW estimator.

In [ ]:
def crossfit_aipw(df, n_splits=5, seed=123, use_forest=True):
    W_cols = ['W1', 'W2', 'W3']
    n = len(df)
    q1 = np.zeros(n)
    q0 = np.zeros(n)
    g = np.zeros(n)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for train_idx, test_idx in kf.split(df):
        train = df.iloc[train_idx]
        test = df.iloc[test_idx]

        if use_forest:
            q_model = RandomForestRegressor(
                n_estimators=200,
                min_samples_leaf=10,
                random_state=seed,
            )
            g_model = RandomForestClassifier(
                n_estimators=200,
                min_samples_leaf=10,
                random_state=seed,
            )
        else:
            q_model = LinearRegression()
            g_model = LogisticRegression(max_iter=1000)

        q_model.fit(train[['A'] + W_cols], train['Y'])
        g_model.fit(train[W_cols], train['A'])

        test1 = test[W_cols].copy()
        test1.insert(0, 'A', 1)
        test0 = test[W_cols].copy()
        test0.insert(0, 'A', 0)

        q1[test_idx] = q_model.predict(test1)
        q0[test_idx] = q_model.predict(test0)
        g[test_idx] = g_model.predict_proba(test[W_cols])[:, 1]

    g = np.clip(g, 0.01, 0.99)
    A = df['A'].to_numpy()
    Y = df['Y'].to_numpy()
    phi = q1 - q0 + A * (Y - q1) / g - (1 - A) * (Y - q0) / (1 - g)
    return phi.mean(), phi.std(ddof=1) / np.sqrt(n)

cf_est, cf_se = crossfit_aipw(df, n_splits=5, use_forest=True)
print(f'Cross-fitted AIPW estimate: {cf_est:.3f}')
print(f'Cross-fitted AIPW 95% CI: ({cf_est - 1.96 * cf_se:.3f}, {cf_est + 1.96 * cf_se:.3f})')

## Exercises

1. Increase the coefficient of `W1` in the treatment mechanism and inspect positivity.
2. Compare IPW with and without propensity-score truncation.
3. Replace the outcome model with a deliberately misspecified linear model.
4. Run the cross-fitted estimator with `use_forest=False` and compare results.
5. Increase the sample size and study how the estimators change.
